# Multi-GPU Training: DDP & FSDP Experiments
## Power of Parallel Computing in Thoracic Disease Classification
**Team 13** | CSYE7105 High Performance Parallel ML & AI

Run this notebook in Open OnDemand Jupyter with **4 GPUs** allocated on `courses-gpu` partition.

## 0. Environment Check

In [ ]:
# Environment Check for Multi-GPU Training
import torch
import os
import sys

# Directory and GPU info
os.chdir('/scratch/dai.zheny/chest_xray_parallel')
print(f"Working dir: {os.getcwd()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | {props.total_memory / 1e9:.1f} GB")

Working dir: /scratch/dai.zheny/chest_xray_parallel
PyTorch version: 2.5.1+cu121
CUDA available: True
CUDA version: 12.1
Number of GPUs: 4
  GPU 0: Tesla P100-PCIE-12GB | 12.8 GB
  GPU 1: Tesla P100-PCIE-12GB | 12.8 GB
  GPU 2: Tesla P100-PCIE-12GB | 12.8 GB
  GPU 3: Tesla P100-PCIE-12GB | 12.8 GB


## 1. Imports & Shared Utilities

In [ ]:
# ============================================================
# Imports for distributed training
# ============================================================
import time
import json
import functools

import torch
import torch.nn as nn
# Distributed primitives: process groups, all_reduce, barriers
import torch.distributed as dist
# Used to spawn one process per GPU
import torch.multiprocessing as mp
# DDP: every GPU keeps a full model copy; gradients are synced each step
from torch.nn.parallel import DistributedDataParallel as DDP
# FSDP: model parameters are sharded across GPUs to save memory
from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    MixedPrecision,
    ShardingStrategy,
)
from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
# Mixed precision (FP16) for faster compute and lower memory
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import roc_auc_score
import numpy as np

# Project files (model and dataset are shared with single-GPU training)
from model import get_model, count_parameters
from dataset import get_dataloaders, DISEASE_LABELS

print("All imports successful!")

In [3]:
# ============== Shared Config ==============
DATA_DIR = './data'
CSV_PATH = './data/Data_Entry_2017_v2020.csv'
OUTPUT_DIR = './results'
EPOCHS = 5
BATCH_SIZE = 32
LR = 1e-4
NUM_WORKERS = 4
IMG_SIZE = 224
USE_PROCESSED = False

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Config ready. Output dir: {OUTPUT_DIR}")

Config ready. Output dir: ./results


In [ ]:
# ============================================================
# Shared utilities for distributed training
# ============================================================

def setup_dist(rank, world_size, port):
    """
    Initialize the distributed process group for one worker.

    rank: ID of this process (0 .. world_size-1)
    world_size: total number of GPUs / processes
    port: TCP port used as the rendezvous master
    """
    # All processes connect to the same address/port to form a group
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = str(port)
    # NCCL is NVIDIA's library for fast GPU-to-GPU communication
    dist.init_process_group(backend='nccl', rank=rank, world_size=world_size)
    # Bind this process to its own GPU so CUDA calls go to the right device
    torch.cuda.set_device(rank)


def cleanup_dist():
    """Destroy the process group after training finishes."""
    dist.destroy_process_group()


def train_one_epoch_dist(model, loader, criterion, optimizer, device, epoch, rank,
                         use_amp=False, scaler=None, sampler=None):
    """One distributed training epoch. Runs on every GPU process."""
    model.train()
    # Reshuffle data differently each epoch across all GPUs.
    if sampler is not None:
        sampler.set_epoch(epoch)

    total_loss = 0.0
    num_batches = 0
    num_samples = 0
    epoch_start = time.time()

    for batch_idx, (images, labels) in enumerate(loader):
        # non_blocking=True: copy CPU->GPU asynchronously so it can overlap
        # with GPU compute (works together with pin_memory=True in DataLoader)
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # set_to_none=True: skip the grad-filling kernel.
        # Slightly faster and uses a bit less memory than zero_grad().
        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            # autocast: run forward ops in FP16 where safe, FP32 otherwise
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            # GradScaler scales up the loss so small FP16 gradients don't
            # underflow to zero, then unscales before optimizer.step().
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            # DDP automatically all-reduces gradients across GPUs during backward
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        num_batches += 1
        num_samples += images.size(0)

        # Only rank 0 prints, otherwise logs would be duplicated per-GPU
        if batch_idx % 50 == 0 and rank == 0:
            elapsed = time.time() - epoch_start
            throughput = num_samples / elapsed if elapsed > 0 else 0
            gpu_mem = torch.cuda.memory_allocated(device) / 1e9
            print(f"  Epoch {epoch} | Batch {batch_idx}/{len(loader)} | "
                  f"Loss: {loss.item():.4f} | "
                  f"Throughput: {throughput:.0f} img/s (rank0) | "
                  f"GPU Mem: {gpu_mem:.2f}GB")

    epoch_time = time.time() - epoch_start

    # Sum loss / batch counts across GPUs so the reported loss is global
    loss_tensor = torch.tensor([total_loss, num_batches], device=device, dtype=torch.float64)
    dist.all_reduce(loss_tensor, op=dist.ReduceOp.SUM)
    avg_loss = loss_tensor[0].item() / loss_tensor[1].item()

    # Total throughput = per-GPU samples * number of GPUs / time
    world_size = dist.get_world_size()
    total_throughput = num_samples * world_size / epoch_time

    return avg_loss, epoch_time, total_throughput


def evaluate_dist(model, loader, criterion, device, use_amp=False):
    """Evaluate the model. Each rank evaluates its shard of the val/test set."""
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            if use_amp:
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)
            total_loss += loss.item()
            # sigmoid -> per-class probability (multi-label, classes are independent)
            probs = torch.sigmoid(outputs)
            all_preds.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # One AUC per disease. Wrapped in try/except because a shard may contain
    # only one class value for a rare disease, which makes roc_auc_score raise.
    aucs = {}
    for i, label_name in enumerate(DISEASE_LABELS):
        try:
            auc = roc_auc_score(all_labels[:, i], all_preds[:, i])
            aucs[label_name] = auc
        except ValueError:
            aucs[label_name] = 0.0

    mean_auc = np.mean(list(aucs.values()))
    return avg_loss, mean_auc, aucs


print("Shared functions defined!")

---
## 2. DDP (Distributed Data Parallel)

In [ ]:
# ============================================================
# DDP (Distributed Data Parallel) Worker
# ------------------------------------------------------------
# Each GPU holds a full copy of the model. Gradients are
# all-reduced across GPUs every backward pass, so all copies
# stay in sync. Simple, fast, but memory-heavy for big models.

def ddp_train_loop(rank, world_size, use_amp=False):
    """Run a full DDP training (all epochs + test eval) on one GPU/rank."""
    device = torch.device(f'cuda:{rank}')
    tag = 'FP16' if use_amp else 'FP32'

    if rank == 0:
        print(f"\n{'='*60}")
        print(f"DDP Training: {tag} | GPUs: {world_size}")
        print(f"{'='*60}")

    # distributed=True makes get_dataloaders use DistributedSampler,
    # which gives each GPU a non-overlapping slice of the data.
    train_loader, val_loader, test_loader, train_sampler = get_dataloaders(
        DATA_DIR, CSV_PATH,
        batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
        img_size=IMG_SIZE, use_processed=USE_PROCESSED, distributed=True,
    )

    # Build model on this rank's GPU, then wrap with DDP.
    model = get_model(num_classes=14, pretrained=True).to(device)
    # device_ids tells DDP which GPU this process owns.
    model = DDP(model, device_ids=[rank], output_device=rank)

    if rank == 0:
        # .module unwraps the DDP wrapper to access the underlying model.
        total_p, train_p = count_parameters(model.module)
        print(f"Model params: {total_p:,} | Trainable: {train_p:,}")
        # Effective batch = batch per GPU * number of GPUs.
        print(f"Batch per GPU: {BATCH_SIZE} | Effective: {BATCH_SIZE * world_size}")

    criterion = nn.BCEWithLogitsLoss()
    optimizer = Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
    # GradScaler is only used for FP16 mixed precision.
    scaler = GradScaler() if use_amp else None

    metrics = {'train_loss': [], 'val_auc': [], 'epoch_times': [],
               'throughputs': [], 'gpu_memory': []}
    total_time = 0.0

    for epoch in range(1, EPOCHS + 1):
        if rank == 0:
            print(f"\n--- DDP {tag} Epoch {epoch}/{EPOCHS} ---")
        # Reset peak tracker so each epoch reports its own peak memory.
        torch.cuda.reset_peak_memory_stats(device)

        train_loss, epoch_time, throughput = train_one_epoch_dist(
            model, train_loader, criterion, optimizer, device, epoch, rank,
            use_amp=use_amp, scaler=scaler, sampler=train_sampler,
        )
        val_loss, val_auc, _ = evaluate_dist(model, val_loader, criterion, device, use_amp=use_amp)
        scheduler.step()

        peak_mem = torch.cuda.max_memory_allocated(device) / 1e9
        total_time += epoch_time
        metrics['train_loss'].append(train_loss)
        metrics['val_auc'].append(val_auc)
        metrics['epoch_times'].append(epoch_time)
        metrics['throughputs'].append(throughput)
        metrics['gpu_memory'].append(peak_mem)

        if rank == 0:
            print(f"  Loss: {train_loss:.4f} | Val AUC: {val_auc:.4f} | "
                  f"Time: {epoch_time:.1f}s | Throughput: {throughput:.0f} img/s | "
                  f"Peak GPU: {peak_mem:.2f}GB")

    test_loss, test_auc, test_aucs = evaluate_dist(model, test_loader, criterion, device, use_amp=use_amp)
    if rank == 0:
        print(f"\nDDP {tag} Test AUC: {test_auc:.4f}")

    return {
        'total_time': total_time,
        'avg_epoch_time': total_time / EPOCHS,
        'test_auc': test_auc,
        'test_aucs': test_aucs,
        'metrics': metrics,
    }


def ddp_worker(rank, world_size, port, result_dict):
    """
    Entry point for each spawned process.
    Runs FP32 first, then FP16, and has rank 0 collect the final results.
    """
    setup_dist(rank, world_size, port)

    fp32_results = ddp_train_loop(rank, world_size, use_amp=False)
    # Wait for every rank to finish FP32 before starting FP16.
    dist.barrier()
    fp16_results = ddp_train_loop(rank, world_size, use_amp=True)

    # Only rank 0 writes to the shared result dict (avoids races / duplicates).
    if rank == 0:
        speedup = (fp32_results['avg_epoch_time'] / fp16_results['avg_epoch_time']
                   if fp16_results['avg_epoch_time'] > 0 else 0)
        result_dict['results'] = {
            'mode': 'DDP',
            'num_gpus': world_size,
            'gpu': torch.cuda.get_device_name(rank),
            'epochs': EPOCHS,
            'batch_size_per_gpu': BATCH_SIZE,
            'effective_batch_size': BATCH_SIZE * world_size,
            'num_workers': NUM_WORKERS,
            'fp32': fp32_results,
            'fp16': fp16_results,
            'speedup_fp16_over_fp32': speedup,
        }

        print(f"\n{'='*60}")
        print(f"DDP {world_size}-GPU SUMMARY")
        print(f"{'='*60}")
        print(f"{'Metric':<30} {'FP32':<15} {'FP16':<15}")
        print(f"{'-'*60}")
        print(f"{'Avg epoch time (s)':<30} {fp32_results['avg_epoch_time']:<15.2f} {fp16_results['avg_epoch_time']:<15.2f}")
        print(f"{'Avg throughput (img/s)':<30} {np.mean(fp32_results['metrics']['throughputs']):<15.0f} {np.mean(fp16_results['metrics']['throughputs']):<15.0f}")
        print(f"{'Peak GPU memory (GB)':<30} {max(fp32_results['metrics']['gpu_memory']):<15.2f} {max(fp16_results['metrics']['gpu_memory']):<15.2f}")
        print(f"{'Test AUC':<30} {fp32_results['test_auc']:<15.4f} {fp16_results['test_auc']:<15.4f}")

    cleanup_dist()


def run_ddp(num_gpus):
    """Launch a DDP experiment with `num_gpus` processes and return results."""
    world_size = min(num_gpus, torch.cuda.device_count())
    print(f"\nLaunching DDP with {world_size} GPUs...")
    # Manager dict lets rank 0 pass results back to the main (parent) process.
    manager = mp.Manager()
    result_dict = manager.dict()
    # Unique port per experiment so repeated runs don't collide.
    port = 29500 + world_size
    # mp.spawn starts one process per GPU, each running ddp_worker.
    mp.spawn(ddp_worker, args=(world_size, port, result_dict), nprocs=world_size, join=True)

    results = dict(result_dict['results'])
    output_file = os.path.join(OUTPUT_DIR, f'ddp_{world_size}gpu_results.json')
    with open(output_file, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"Results saved to {output_file}")
    return results


print("DDP functions defined!")

### 2a. DDP 2-GPU

In [3]:
import subprocess
PYTHON = '/home/dai.zheny/.conda/envs/chest_xray/bin/python'
DIR = '/scratch/dai.zheny/chest_xray_parallel'

subprocess.run([PYTHON, f'{DIR}/train_ddp.py', '--num_gpus', '2', '--data_dir', f'{DIR}/data', '--csv_path', f'{DIR}/data/Data_Entry_2017_v2020.csv', '--epochs', '5', '--batch_size', '32', '--lr', '1e-4', '--num_workers', '2', '--output_dir', f'{DIR}/results'], cwd=DIR)

Launching DDP with 2 GPUs
Available GPUs: 4
  GPU 0: Tesla P100-PCIE-12GB
  GPU 1: Tesla P100-PCIE-12GB
  GPU 2: Tesla P100-PCIE-12GB
  GPU 3: Tesla P100-PCIE-12GB
[TRAIN] Loaded 69219 images
[VAL] Loaded 17305 images
[TEST] Loaded 25596 images

DDP Training: FP32 | GPUs: 2
[TRAIN] Loaded 69219 images
[VAL] Loaded 17305 images
[TEST] Loaded 25596 images
Model: DenseNet-121 | Params: 7,485,838 | Trainable: 7,485,838
Batch size per GPU: 32 | Effective batch size: 64

--- DDP FP32 Epoch 1/5 ---
  Epoch 1 | Batch 0/1081 | Loss: 0.6728 | Throughput: 2 img/s (rank0) | GPU Mem: 0.19GB
  Epoch 1 | Batch 50/1081 | Loss: 0.1127 | Throughput: 26 img/s (rank0) | GPU Mem: 0.19GB
  Epoch 1 | Batch 100/1081 | Loss: 0.1760 | Throughput: 28 img/s (rank0) | GPU Mem: 0.19GB
  Epoch 1 | Batch 150/1081 | Loss: 0.1225 | Throughput: 29 img/s (rank0) | GPU Mem: 0.19GB
  Epoch 1 | Batch 200/1081 | Loss: 0.1854 | Throughput: 29 img/s (rank0) | GPU Mem: 0.19GB
  Epoch 1 | Batch 250/1081 | Loss: 0.1545 | Throughp

/scratch/dai.zheny/chest_xray_parallel/train_ddp.py:163: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if use_amp else None
/scratch/dai.zheny/chest_xray_parallel/train_ddp.py:163: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if use_amp else None


[TRAIN] Loaded 69219 images
[VAL] Loaded 17305 images
[TEST] Loaded 25596 images

DDP FP32 Test AUC: 0.8029

DDP Training: FP16 | GPUs: 2
[TRAIN] Loaded 69219 images
[VAL] Loaded 17305 images
[TEST] Loaded 25596 images
Model: DenseNet-121 | Params: 7,485,838 | Trainable: 7,485,838
Batch size per GPU: 32 | Effective batch size: 64

--- DDP FP16 Epoch 1/5 ---


/scratch/dai.zheny/chest_xray_parallel/train_ddp.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/scratch/dai.zheny/chest_xray_parallel/train_ddp.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Epoch 1 | Batch 0/1081 | Loss: 0.7008 | Throughput: 2 img/s (rank0) | GPU Mem: 0.19GB
  Epoch 1 | Batch 50/1081 | Loss: 0.1127 | Throughput: 26 img/s (rank0) | GPU Mem: 0.19GB
  Epoch 1 | Batch 100/1081 | Loss: 0.1749 | Throughput: 28 img/s (rank0) | GPU Mem: 0.19GB
  Epoch 1 | Batch 150/1081 | Loss: 0.1242 | Throughput: 29 img/s (rank0) | GPU Mem: 0.19GB
  Epoch 1 | Batch 200/1081 | Loss: 0.1983 | Throughput: 29 img/s (rank0) | GPU Mem: 0.19GB
  Epoch 1 | Batch 250/1081 | Loss: 0.1633 | Throughput: 30 img/s (rank0) | GPU Mem: 0.19GB
  Epoch 1 | Batch 300/1081 | Loss: 0.1760 | Throughput: 30 img/s (rank0) | GPU Mem: 0.19GB
  Epoch 1 | Batch 350/1081 | Loss: 0.1926 | Throughput: 30 img/s (rank0) | GPU Mem: 0.19GB
  Epoch 1 | Batch 400/1081 | Loss: 0.1302 | Throughput: 30 img/s (rank0) | GPU Mem: 0.19GB
  Epoch 1 | Batch 450/1081 | Loss: 0.1582 | Throughput: 30 img/s (rank0) | GPU Mem: 0.19GB
  Epoch 1 | Batch 500/1081 | Loss: 0.2018 | Throughput: 30 img/s (rank0) | GPU Mem: 0.19GB
  E

...

/scratch/dai.zheny/chest_xray_parallel/train_ddp.py:107: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/scratch/dai.zheny/chest_xray_parallel/train_ddp.py:107: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Loss: 0.1242 | Val AUC: 0.8363 | Time: 1130.6s | Throughput: 61 img/s | Peak GPU: 2.32GB


/scratch/dai.zheny/chest_xray_parallel/train_ddp.py:107: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/scratch/dai.zheny/chest_xray_parallel/train_ddp.py:107: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



DDP FP16 Test AUC: 0.8034

DDP 2-GPU SUMMARY
Metric                         FP32            FP16           
------------------------------------------------------------
Avg epoch time (s)             1134.97         1125.44        
Avg throughput (img/s)         61              61             
Peak GPU memory (GB)           4.36            2.32           
Test AUC                       0.8029          0.8034         

Results saved to /scratch/dai.zheny/chest_xray_parallel/results/ddp_2gpu_results.json


CompletedProcess(args=['/home/dai.zheny/.conda/envs/chest_xray/bin/python', '/scratch/dai.zheny/chest_xray_parallel/train_ddp.py', '--num_gpus', '2', '--data_dir', '/scratch/dai.zheny/chest_xray_parallel/data', '--csv_path', '/scratch/dai.zheny/chest_xray_parallel/data/Data_Entry_2017_v2020.csv', '--epochs', '5', '--batch_size', '32', '--lr', '1e-4', '--num_workers', '2', '--output_dir', '/scratch/dai.zheny/chest_xray_parallel/results'], returncode=0)

### 2b. DDP 4-GPU

In [8]:
import subprocess
subprocess.run([PYTHON, f'{DIR}/train_ddp.py', '--num_gpus', '4', '--data_dir', f'{DIR}/data', '--csv_path', f'{DIR}/data/Data_Entry_2017_v2020.csv', '--epochs', '5', '--batch_size', '32', '--lr', '1e-4', '--num_workers', '2', '--output_dir', f'{DIR}/results'], cwd=DIR)

Launching DDP with 4 GPUs
Available GPUs: 4
  GPU 0: Tesla P100-PCIE-12GB
  GPU 1: Tesla P100-PCIE-12GB
  GPU 2: Tesla P100-PCIE-12GB
  GPU 3: Tesla P100-PCIE-12GB


[W412 00:27:23.487062709 socket.cpp:752] [c10d] The client socket has failed to connect to [localhost]:29500 (errno: 99 - Cannot assign requested address).


[TRAIN] Loaded 69219 images
[VAL] Loaded 17305 images
[TEST] Loaded 25596 images
[TRAIN] Loaded 69219 images
[VAL] Loaded 17305 images
[TEST] Loaded 25596 images

DDP Training: FP32 | GPUs: 4
[TRAIN] Loaded 69219 images
[VAL] Loaded 17305 images
[TEST] Loaded 25596 images
Model: DenseNet-121 | Params: 7,485,838 | Trainable: 7,485,838
Batch size per GPU: 32 | Effective batch size: 128

--- DDP FP32 Epoch 1/5 ---
[TRAIN] Loaded 69219 images
[VAL] Loaded 17305 images
[TEST] Loaded 25596 images
  Epoch 1 | Batch 0/540 | Loss: 0.6854 | Throughput: 2 img/s (rank0) | GPU Mem: 0.19GB
  Epoch 1 | Batch 50/540 | Loss: 0.1596 | Throughput: 22 img/s (rank0) | GPU Mem: 0.19GB
  Epoch 1 | Batch 100/540 | Loss: 0.1595 | Throughput: 24 img/s (rank0) | GPU Mem: 0.19GB
  Epoch 1 | Batch 150/540 | Loss: 0.1180 | Throughput: 24 img/s (rank0) | GPU Mem: 0.19GB
  Epoch 1 | Batch 200/540 | Loss: 0.1220 | Throughput: 24 img/s (rank0) | GPU Mem: 0.19GB
  Epoch 1 | Batch 250/540 | Loss: 0.1947 | Throughput: 25 

/scratch/dai.zheny/chest_xray_parallel/train_ddp.py:163: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if use_amp else None
/scratch/dai.zheny/chest_xray_parallel/train_ddp.py:163: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if use_amp else None
/scratch/dai.zheny/chest_xray_parallel/train_ddp.py:163: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if use_amp else None
/scratch/dai.zheny/chest_xray_parallel/train_ddp.py:163: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if use_amp else None


[TRAIN] Loaded 69219 images
[VAL] Loaded 17305 images
[TEST] Loaded 25596 images
[TRAIN] Loaded 69219 images
[VAL] Loaded 17305 images
[TEST] Loaded 25596 images
[TRAIN] Loaded 69219 images
[VAL] Loaded 17305 images
[TEST] Loaded 25596 images

DDP FP32 Test AUC: 0.7966

DDP Training: FP16 | GPUs: 4
[TRAIN] Loaded 69219 images
[VAL] Loaded 17305 images
[TEST] Loaded 25596 images
Model: DenseNet-121 | Params: 7,485,838 | Trainable: 7,485,838
Batch size per GPU: 32 | Effective batch size: 128

--- DDP FP16 Epoch 1/5 ---


...

/scratch/dai.zheny/chest_xray_parallel/train_ddp.py:107: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/scratch/dai.zheny/chest_xray_parallel/train_ddp.py:107: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/scratch/dai.zheny/chest_xray_parallel/train_ddp.py:107: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/scratch/dai.zheny/chest_xray_parallel/train_ddp.py:107: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Loss: 0.1263 | Val AUC: 0.8316 | Time: 593.4s | Throughput: 116 img/s | Peak GPU: 2.32GB


/scratch/dai.zheny/chest_xray_parallel/train_ddp.py:107: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/scratch/dai.zheny/chest_xray_parallel/train_ddp.py:107: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/scratch/dai.zheny/chest_xray_parallel/train_ddp.py:107: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/scratch/dai.zheny/chest_xray_parallel/train_ddp.py:107: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



DDP FP16 Test AUC: 0.7936

DDP 4-GPU SUMMARY
Metric                         FP32            FP16           
------------------------------------------------------------
Avg epoch time (s)             603.47          596.65         
Avg throughput (img/s)         115             116            
Peak GPU memory (GB)           4.36            2.32           
Test AUC                       0.7966          0.7936         

Results saved to /scratch/dai.zheny/chest_xray_parallel/results/ddp_4gpu_results.json


CompletedProcess(args=['/home/dai.zheny/.conda/envs/chest_xray/bin/python', '/scratch/dai.zheny/chest_xray_parallel/train_ddp.py', '--num_gpus', '4', '--data_dir', '/scratch/dai.zheny/chest_xray_parallel/data', '--csv_path', '/scratch/dai.zheny/chest_xray_parallel/data/Data_Entry_2017_v2020.csv', '--epochs', '5', '--batch_size', '32', '--lr', '1e-4', '--num_workers', '2', '--output_dir', '/scratch/dai.zheny/chest_xray_parallel/results'], returncode=0)

---
## 3. FSDP (Fully Sharded Data Parallel)

In [ ]:
# FSDP (Fully Sharded Data Parallel) Worker
# ------------------------------------------------------------
# Unlike DDP, FSDP shards the model parameters themselves across
# GPUs. Each GPU only holds a slice; parameters are gathered on
# the fly for forward/backward. This saves memory (so we can fit
# bigger models or batches) at the cost of extra communication.

def get_fsdp_model(model, device, use_amp=False):
    """Wrap a plain model in FSDP with our sharding / precision policy."""
    # Auto-wrap: FSDP creates a shard for any submodule with >=100k params.
    # Smaller modules are cheaper to keep unsharded.
    auto_wrap_policy = functools.partial(
        size_based_auto_wrap_policy,
        min_num_params=100_000,
    )
    # Mixed precision is configured directly on FSDP.
    mixed_precision_policy = None
    if use_amp:
        mixed_precision_policy = MixedPrecision(
            param_dtype=torch.float16,   # parameters stored/compute in FP16
            reduce_dtype=torch.float16,  # gradient all-reduce in FP16
            buffer_dtype=torch.float16,  # buffers (e.g. BN stats) in FP16
        )
    fsdp_model = FSDP(
        model,
        auto_wrap_policy=auto_wrap_policy,
        mixed_precision=mixed_precision_policy,
        # FULL_SHARD: shard params, gradients, and optimizer states.
        # Most memory-efficient option, comparable to ZeRO-3.
        sharding_strategy=ShardingStrategy.FULL_SHARD,
        device_id=device,
    )
    return fsdp_model


def fsdp_train_loop(rank, world_size, use_amp=False):
    """Run a full FSDP training (all epochs + test eval) on one rank."""
    device = torch.device(f'cuda:{rank}')
    tag = 'FP16' if use_amp else 'FP32'

    if rank == 0:
        print(f"\n{'='*60}")
        print(f"FSDP Training: {tag} | GPUs: {world_size}")
        print(f"{'='*60}")

    train_loader, val_loader, test_loader, train_sampler = get_dataloaders(
        DATA_DIR, CSV_PATH,
        batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
        img_size=IMG_SIZE, use_processed=USE_PROCESSED, distributed=True,
    )

    # Build the model on CPU first; FSDP shards/moves it to `device` for us.
    model = get_model(num_classes=14, pretrained=True)

    if rank == 0:
        total_p, train_p = count_parameters(model)
        print(f"Model params: {total_p:,} | Trainable: {train_p:,}")
        print(f"Batch per GPU: {BATCH_SIZE} | Effective: {BATCH_SIZE * world_size}")
        print(f"Sharding: FULL_SHARD")

    model = get_fsdp_model(model, device, use_amp=use_amp)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

    metrics = {'train_loss': [], 'val_auc': [], 'epoch_times': [],
               'throughputs': [], 'gpu_memory': []}
    total_time = 0.0

    for epoch in range(1, EPOCHS + 1):
        if rank == 0:
            print(f"\n--- FSDP {tag} Epoch {epoch}/{EPOCHS} ---")
        torch.cuda.reset_peak_memory_stats(device)

        # FSDP handles mixed precision via its MixedPrecision policy,
        # so we do NOT pass use_amp/scaler down to the training loop.
        train_loss, epoch_time, throughput = train_one_epoch_dist(
            model, train_loader, criterion, optimizer, device, epoch, rank,
            use_amp=False, scaler=None, sampler=train_sampler,
        )
        val_loss, val_auc, _ = evaluate_dist(model, val_loader, criterion, device, use_amp=False)
        scheduler.step()

        peak_mem = torch.cuda.max_memory_allocated(device) / 1e9
        total_time += epoch_time
        metrics['train_loss'].append(train_loss)
        metrics['val_auc'].append(val_auc)
        metrics['epoch_times'].append(epoch_time)
        metrics['throughputs'].append(throughput)
        metrics['gpu_memory'].append(peak_mem)

        if rank == 0:
            print(f"  Loss: {train_loss:.4f} | Val AUC: {val_auc:.4f} | "
                  f"Time: {epoch_time:.1f}s | Throughput: {throughput:.0f} img/s | "
                  f"Peak GPU: {peak_mem:.2f}GB")

    test_loss, test_auc, test_aucs = evaluate_dist(model, test_loader, criterion, device, use_amp=False)
    if rank == 0:
        print(f"\nFSDP {tag} Test AUC: {test_auc:.4f}")

    return {
        'total_time': total_time,
        'avg_epoch_time': total_time / EPOCHS,
        'test_auc': test_auc,
        'test_aucs': test_aucs,
        'metrics': metrics,
    }


def fsdp_worker(rank, world_size, port, result_dict):
    """Entry point for each FSDP process. Runs FP32 then FP16."""
    setup_dist(rank, world_size, port)

    fp32_results = fsdp_train_loop(rank, world_size, use_amp=False)
    dist.barrier()
    fp16_results = fsdp_train_loop(rank, world_size, use_amp=True)

    if rank == 0:
        speedup = (fp32_results['avg_epoch_time'] / fp16_results['avg_epoch_time']
                   if fp16_results['avg_epoch_time'] > 0 else 0)
        result_dict['results'] = {
            'mode': 'FSDP',
            'num_gpus': world_size,
            'gpu': torch.cuda.get_device_name(rank),
            'epochs': EPOCHS,
            'batch_size_per_gpu': BATCH_SIZE,
            'effective_batch_size': BATCH_SIZE * world_size,
            'num_workers': NUM_WORKERS,
            'sharding_strategy': 'FULL_SHARD',
            'fp32': fp32_results,
            'fp16': fp16_results,
            'speedup_fp16_over_fp32': speedup,
        }

        print(f"\n{'='*60}")
        print(f"FSDP {world_size}-GPU SUMMARY")
        print(f"{'='*60}")
        print(f"{'Metric':<30} {'FP32':<15} {'FP16':<15}")
        print(f"{'-'*60}")
        print(f"{'Avg epoch time (s)':<30} {fp32_results['avg_epoch_time']:<15.2f} {fp16_results['avg_epoch_time']:<15.2f}")
        print(f"{'Avg throughput (img/s)':<30} {np.mean(fp32_results['metrics']['throughputs']):<15.0f} {np.mean(fp16_results['metrics']['throughputs']):<15.0f}")
        print(f"{'Peak GPU memory (GB)':<30} {max(fp32_results['metrics']['gpu_memory']):<15.2f} {max(fp16_results['metrics']['gpu_memory']):<15.2f}")
        print(f"{'Test AUC':<30} {fp32_results['test_auc']:<15.4f} {fp16_results['test_auc']:<15.4f}")

    cleanup_dist()


def run_fsdp(num_gpus):
    """Launch an FSDP experiment with `num_gpus` processes and return results."""
    world_size = min(num_gpus, torch.cuda.device_count())
    print(f"\nLaunching FSDP with {world_size} GPUs...")
    manager = mp.Manager()
    result_dict = manager.dict()
    # Different port range from DDP so both can run back-to-back cleanly.
    port = 29600 + world_size
    mp.spawn(fsdp_worker, args=(world_size, port, result_dict), nprocs=world_size, join=True)

    results = dict(result_dict['results'])
    output_file = os.path.join(OUTPUT_DIR, f'fsdp_{world_size}gpu_results.json')
    with open(output_file, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"Results saved to {output_file}")
    return results


print("FSDP functions defined!")

### 3a. FSDP 2-GPU

In [1]:
# FSDP 2-GPU
subprocess.run([PYTHON, f'{DIR}/train_fsdp.py', '--num_gpus', '2', '--data_dir', f'{DIR}/data', '--csv_path', f'{DIR}/data/Data_Entry_2017_v2020.csv', '--epochs', '5', '--batch_size', '32', '--lr', '1e-4', '--num_workers', '2', '--output_dir', f'{DIR}/results'], cwd=DIR)

Launching FSDP with 2 GPUs
Available GPUs: 4
  GPU 0: Tesla P100-PCIE-12GB
  GPU 1: Tesla P100-PCIE-12GB
  GPU 2: Tesla P100-PCIE-12GB
  GPU 3: Tesla P100-PCIE-12GB


Traceback (most recent call last):
  File "/scratch/dai.zheny/chest_xray_parallel/train_fsdp.py", line 297, in <module>
    main()
  File "/scratch/dai.zheny/chest_xray_parallel/train_fsdp.py", line 293, in main
    mp.spawn(worker, args=(world_size, args), nprocs=world_size, join=True)
  File "/home/dai.zheny/.conda/envs/chest_xray/lib/python3.10/site-packages/torch/multiprocessing/spawn.py", line 328, in spawn
    return start_processes(fn, args, nprocs, join, daemon, start_method="spawn")
  File "/home/dai.zheny/.conda/envs/chest_xray/lib/python3.10/site-packages/torch/multiprocessing/spawn.py", line 284, in start_processes
    while not context.join():
  File "/home/dai.zheny/.conda/envs/chest_xray/lib/python3.10/site-packages/torch/multiprocessing/spawn.py", line 132, in join
    ready = multiprocessing.connection.wait(
  File "/home/dai.zheny/.conda/envs/chest_xray/lib/python3.10/multiprocessing/connection.py", line 931, in wait
    ready = selector.select(timeout)
  File "/home/

KeyboardInterrupt: 

### 3b. FSDP 4-GPU

In [2]:
# FSDP 4-GPU
subprocess.run([PYTHON, f'{DIR}/train_fsdp.py', '--num_gpus', '4', '--data_dir', f'{DIR}/data', '--csv_path', f'{DIR}/data/Data_Entry_2017_v2020.csv', '--epochs', '5', '--batch_size', '32', '--lr', '1e-4', '--num_workers', '2', '--output_dir', f'{DIR}/results'], cwd=DIR)

Launching FSDP with 4 GPUs
Available GPUs: 4
  GPU 0: Tesla P100-PCIE-12GB
  GPU 1: Tesla P100-PCIE-12GB
  GPU 2: Tesla P100-PCIE-12GB
  GPU 3: Tesla P100-PCIE-12GB


[W413 01:09:24.657368616 socket.cpp:752] [c10d] The client socket has failed to connect to [localhost]:29501 (errno: 99 - Cannot assign requested address).
[W413 01:09:24.666778107 socket.cpp:752] [c10d] The client socket has failed to connect to [localhost]:29501 (errno: 99 - Cannot assign requested address).
[W413 01:09:24.737502263 socket.cpp:752] [c10d] The client socket has failed to connect to [localhost]:29501 (errno: 99 - Cannot assign requested address).



FSDP Training: FP32 | GPUs: 4
[TRAIN] Loaded 69219 images
[VAL] Loaded 17305 images
[TEST] Loaded 25596 images
Model: DenseNet-121 | Params: 7,485,838 | Trainable: 7,485,838
Batch size per GPU: 32 | Effective batch size: 128
Sharding strategy: FULL_SHARD

--- FSDP FP32 Epoch 1/5 ---
[TRAIN] Loaded 69219 images
[VAL] Loaded 17305 images
[TEST] Loaded 25596 images
[TRAIN] Loaded 69219 images
[VAL] Loaded 17305 images
[TEST] Loaded 25596 images
[TRAIN] Loaded 69219 images
[VAL] Loaded 17305 images
[TEST] Loaded 25596 images
  Epoch 1 | Batch 0/540 | Loss: 0.6891 | Throughput: 2 img/s (rank0) | GPU Mem: 0.07GB
  Epoch 1 | Batch 50/540 | Loss: 0.1570 | Throughput: 20 img/s (rank0) | GPU Mem: 0.07GB
  Epoch 1 | Batch 100/540 | Loss: 0.1522 | Throughput: 22 img/s (rank0) | GPU Mem: 0.07GB
  Epoch 1 | Batch 150/540 | Loss: 0.1154 | Throughput: 23 img/s (rank0) | GPU Mem: 0.07GB
  Epoch 1 | Batch 200/540 | Loss: 0.1205 | Throughput: 23 img/s (rank0) | GPU Mem: 0.07GB
  Epoch 1 | Batch 250/540 

/home/dai.zheny/.conda/envs/chest_xray/lib/python3.10/site-packages/torch/distributed/fsdp/_wrap_utils.py:118: UserWarning: Both mixed precision and an auto_wrap_policy were specified to FSDP, where the wrapped module has submodules of type:
{<class 'torch.nn.modules.batchnorm.BatchNorm2d'>}
These modules will be wrapped as separate FSDP instacnes with mixed precision disabled.
  warnings.warn(
/home/dai.zheny/.conda/envs/chest_xray/lib/python3.10/site-packages/torch/distributed/fsdp/_wrap_utils.py:118: UserWarning: Both mixed precision and an auto_wrap_policy were specified to FSDP, where the wrapped module has submodules of type:
{<class 'torch.nn.modules.batchnorm.BatchNorm2d'>}
These modules will be wrapped as separate FSDP instacnes with mixed precision disabled.
  warnings.warn(
/home/dai.zheny/.conda/envs/chest_xray/lib/python3.10/site-packages/torch/distributed/fsdp/_wrap_utils.py:118: UserWarning: Both mixed precision and an auto_wrap_policy were specified to FSDP, where the w


FSDP FP32 Test AUC: 0.7969

FSDP Training: FP16 | GPUs: 4
[TRAIN] Loaded 69219 images
[VAL] Loaded 17305 images
[TEST] Loaded 25596 images
Model: DenseNet-121 | Params: 7,485,838 | Trainable: 7,485,838
Batch size per GPU: 32 | Effective batch size: 128
Sharding strategy: FULL_SHARD

--- FSDP FP16 Epoch 1/5 ---
[TRAIN] Loaded 69219 images
[VAL] Loaded 17305 images
[TEST] Loaded 25596 images
[TRAIN] Loaded 69219 images
[VAL] Loaded 17305 images
[TEST] Loaded 25596 images
[TRAIN] Loaded 69219 images
[VAL] Loaded 17305 images
[TEST] Loaded 25596 images


[rank3]:[W413 02:57:35.619139794 PyInterpreter.cpp:260] Warning: Deallocating Tensor that still has live PyObject references.  This probably happened because you took out a weak reference to Tensor and didn't call _fix_weakref() after dereferencing it.  Subsequent accesses to this tensor via the PyObject will now fail. (function decref)
[rank3]:[W413 02:57:35.619384501 PyInterpreter.cpp:260] Warning: Deallocating Tensor that still has live PyObject references.  This probably happened because you took out a weak reference to Tensor and didn't call _fix_weakref() after dereferencing it.  Subsequent accesses to this tensor via the PyObject will now fail. (function decref)
[rank3]:[W413 02:57:35.619438943 PyInterpreter.cpp:260] Warning: Deallocating Tensor that still has live PyObject references.  This probably happened because you took out a weak reference to Tensor and didn't call _fix_weakref() after dereferencing it.  Subsequent accesses to this tensor via the PyObject will now fail. (

  Epoch 1 | Batch 0/540 | Loss: 0.7144 | Throughput: 2 img/s (rank0) | GPU Mem: 0.07GB
  Epoch 1 | Batch 50/540 | Loss: 0.1552 | Throughput: 17 img/s (rank0) | GPU Mem: 0.07GB
  Epoch 1 | Batch 100/540 | Loss: 0.1581 | Throughput: 19 img/s (rank0) | GPU Mem: 0.07GB
  Epoch 1 | Batch 150/540 | Loss: 0.1160 | Throughput: 20 img/s (rank0) | GPU Mem: 0.07GB
  Epoch 1 | Batch 200/540 | Loss: 0.1214 | Throughput: 20 img/s (rank0) | GPU Mem: 0.07GB
  Epoch 1 | Batch 250/540 | Loss: 0.1869 | Throughput: 20 img/s (rank0) | GPU Mem: 0.07GB
  Epoch 1 | Batch 300/540 | Loss: 0.1981 | Throughput: 20 img/s (rank0) | GPU Mem: 0.07GB
  Epoch 1 | Batch 350/540 | Loss: 0.1256 | Throughput: 20 img/s (rank0) | GPU Mem: 0.07GB
  Epoch 1 | Batch 400/540 | Loss: 0.1855 | Throughput: 20 img/s (rank0) | GPU Mem: 0.07GB
  Epoch 1 | Batch 450/540 | Loss: 0.1832 | Throughput: 20 img/s (rank0) | GPU Mem: 0.07GB
  Epoch 1 | Batch 500/540 | Loss: 0.1368 | Throughput: 20 img/s (rank0) | GPU Mem: 0.07GB
  Loss: 0.1608

CompletedProcess(args=['/home/dai.zheny/.conda/envs/chest_xray/bin/python', '/scratch/dai.zheny/chest_xray_parallel/train_fsdp.py', '--num_gpus', '4', '--data_dir', '/scratch/dai.zheny/chest_xray_parallel/data', '--csv_path', '/scratch/dai.zheny/chest_xray_parallel/data/Data_Entry_2017_v2020.csv', '--epochs', '5', '--batch_size', '32', '--lr', '1e-4', '--num_workers', '2', '--output_dir', '/scratch/dai.zheny/chest_xray_parallel/results'], returncode=0)

---
## 4. Results Summary

In [10]:
import os, json, numpy as np
DIR = '/scratch/dai.zheny/chest_xray_parallel/results'

# 单GPU baseline
with open(os.path.join(DIR, 'training_results.json')) as f:
    single_gpu = json.load(f)

# DDP
with open(os.path.join(DIR, 'ddp_2gpu_results.json')) as f:
    ddp_2 = json.load(f)
with open(os.path.join(DIR, 'ddp_4gpu_results.json')) as f:
    ddp_4 = json.load(f)

# FSDP
with open(os.path.join(DIR, 'fsdp_2gpu_results.json')) as f:
    fsdp_2 = json.load(f)
with open(os.path.join(DIR, 'fsdp_4gpu_results.json')) as f:
    fsdp_4 = json.load(f)

# 汇总对比
print(f"{'='*90}")
print(f"{'Config':<25} {'Avg Epoch(s)':<15} {'Throughput':<15} {'GPU Mem(GB)':<15} {'Test AUC':<10}")
print(f"{'-'*90}")

configs = [
    ('Single GPU FP32', single_gpu['fp32']),
    ('Single GPU FP16', single_gpu['fp16']),
    ('DDP 2-GPU FP32', ddp_2['fp32']),
    ('DDP 2-GPU FP16', ddp_2['fp16']),
    ('DDP 4-GPU FP32', ddp_4['fp32']),
    ('DDP 4-GPU FP16', ddp_4['fp16']),
    ('FSDP 2-GPU FP32', fsdp_2['fp32']),
    ('FSDP 2-GPU FP16', fsdp_2['fp16']),
    ('FSDP 4-GPU FP32', fsdp_4['fp32']),
    ('FSDP 4-GPU FP16', fsdp_4['fp16']),
]

for name, d in configs:
    print(f"{name:<25} {d['avg_epoch_time']:<15.2f} {np.mean(d['metrics']['throughputs']):<15.0f} {max(d['metrics']['gpu_memory']):<15.2f} {d['test_auc']:<10.4f}")

print(f"{'='*90}")

Config                    Avg Epoch(s)    Throughput      GPU Mem(GB)     Test AUC  
------------------------------------------------------------------------------------------
Single GPU FP32           327.62          215             4.33            0.8059    
Single GPU FP16           289.57          239             2.41            0.8062    
DDP 2-GPU FP32            1134.97         61              4.36            0.8029    
DDP 2-GPU FP16            1125.44         61              2.32            0.8034    
DDP 4-GPU FP32            603.47          115             4.36            0.7966    
DDP 4-GPU FP16            596.65          116             2.32            0.7936    
FSDP 2-GPU FP32           601.04          115             4.30            0.8068    
FSDP 2-GPU FP16           719.05          96              3.26            0.8030    
FSDP 4-GPU FP32           636.97          109             4.27            0.7969    
FSDP 4-GPU FP16           826.38          84              3

In [ ]:
print("\nAll experiments complete! Results saved in ./results/")
!ls -la ./results/*.json